## Live Yelp Review Extraction and Analysis

In [2]:
import pickle
import pandas as pd
import scipy.sparse as sp
import re
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [3]:
with open("isolation_forest_yelp.pkl", "rb") as f:
    iso_model = pickle.load(f)

with open("tfidf_vectorizer.pkl", "rb") as f:
    tfidf = pickle.load(f)

print("Model and vectorizer loaded")

Model and vectorizer loaded


In [4]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
def preprocess_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^a-z\s]", "", text)

    tokens = text.split()

    cleaned_tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    return " ".join(cleaned_tokens)

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

def create_driver():

    chrome_options = Options()

    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")

    driver = webdriver.Chrome(options=chrome_options)

    return driver

In [16]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def fetch_yelp_reviews(url):

    driver = create_driver()

    reviews = []

    # Yelp pagination offsets
    offsets = [0, 10, 20]

    for offset in offsets:

        page_url = f"{url}?start={offset}"

        driver.get(page_url)

        time.sleep(6)

        elements = driver.find_elements(By.CSS_SELECTOR, "span[lang='en']")

        for el in elements:

            text = el.text.strip()

            if len(text) > 40:
                reviews.append(text)

    driver.quit()

    return reviews[:30]

In [20]:
def analyze_yelp_business(url):

    reviews = fetch_yelp_reviews(url)

    print("Reviews collected:", len(reviews))

    if len(reviews) == 0:
        print("No reviews extracted")
        return

    clean_reviews = [preprocess_text(r) for r in reviews]

    X_text = tfidf.transform(clean_reviews)

    review_length = [[len(r.split())] for r in clean_reviews]

    # placeholder rating (3 stars as neutral value)
    rating_placeholder = [[3] for _ in clean_reviews]

    X_live = sp.hstack([X_text, review_length, rating_placeholder])

    predictions = iso_model.predict(X_live)

    suspicious = [1 if p == -1 else 0 for p in predictions]

    suspicious_percentage = (sum(suspicious) / len(suspicious)) * 100

    print("\nSuspicious review percentage:", round(suspicious_percentage,2), "%")

    if suspicious_percentage > 20:
        print("⚠️ Business may have suspicious review activity")
    else:
        print("✅ Reviews appear mostly genuine")

    for review, pred in zip(reviews, predictions):
        label = "Suspicious" if pred == -1 else "Normal"
        print(f"\n[{label}] {review[:120]}...")

In [21]:
url = "https://www.yelp.com/biz/four-barrel-coffee-san-francisco"

analyze_yelp_business(url)

Reviews collected: 20

Suspicious review percentage: 0.0 %
✅ Reviews appear mostly genuine

[Normal] Very modern and cool place to visit and try really good coffee.

There are small tables for small groups mostly. Dogs fr...

[Normal] This review is just for the pastries and atmosphere, since I didn't get a drink. The salted chocolate babka is yummy, bu...

[Normal] I checked this place out recently and although not really a fan of the location, I was surprised because the interior wa...

[Normal] Coffee was not very good. There is water leaking from the windows by the seats in the front of the...

[Normal] A nice ambience - very light and open. However, there is no WiFi available (which is why I took off a star), so take tha...

[Normal] Coffee is hit or miss. But the last time I went, I got an iced latte that tasted amazing. The coffee was fragrant and it...

[Normal] We were apartment hunting in the Mission District and wanted to rest our tired feet, use a clean bathroom, and get so